In [1]:
pip install openai pypdf tqdm numpy


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import numpy as np
from uuid import uuid4
from tqdm import tqdm
from pypdf import PdfReader
from openai import OpenAI

# ---------------- CONFIG ----------------
DATA_DIR = r"c:\Users\MSI\industrix\industrix_1"
OUTPUT_FILE = "embeddings3.jsonl"

EMBED_MODEL = "text-embedding-3-large"  # 3072 dim
CHUNK_SIZE = 400
OVERLAP = 50
BATCH_SIZE = 64
# ---------------------------------------

client = OpenAI(api_key="")

# -------- TEXT LOADERS --------
def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append((i + 1, text))
    return pages

def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return [(None, f.read())]

# -------- CHUNKING --------
def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
    return chunks

# -------- OPENAI EMBEDDINGS --------
def embed_texts(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [
        np.array(e.embedding, dtype="float32").tolist()
        for e in response.data
    ]

# -------- PROCESSING --------
def process_file(path, out_f):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        pages = load_pdf(path)
    elif ext == ".txt":
        pages = load_txt(path)
    else:
        return 0

    total_chunks = 0

    for page, text in pages:
        chunks = chunk_text(text, CHUNK_SIZE, OVERLAP)
        if not chunks:
            continue

        # батчами — дешевле и стабильнее
        for i in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[i:i + BATCH_SIZE]
            vectors = embed_texts(batch)

            for chunk, vector in zip(batch, vectors):
                record = {
                    "id": str(uuid4()),
                    "vector": vector,
                    "payload": {
                        "text": chunk,
                        "source": path,
                        "page": page,
                        "model": EMBED_MODEL
                    }
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                total_chunks += 1

    return total_chunks

# -------- MAIN --------
def main():
    files = []
    for root, _, filenames in os.walk(DATA_DIR):
        for name in filenames:
            if name.endswith((".pdf", ".txt")):
                files.append(os.path.join(root, name))

    print(f"Найдено файлов: {len(files)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
        for path in tqdm(files):
            try:
                chunks = process_file(path, out_f)
                tqdm.write(f"{path} → {chunks} chunks")
            except Exception as e:
                tqdm.write(f"❌ Ошибка {path}: {e}")

    print(f"✅ Готово. Результат: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


Найдено файлов: 11


  0%|          | 0/11 [00:00<?, ?it/s]Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 214 0 (offset 0)
Ignoring wrong pointing object 220 0 (offset 0)
Ignoring wrong pointing object 299 0 (offset 0)
Ignoring wrong pointing object 337 0 (offset 0)
Ignoring wrong pointing object 377 0 (offset 0)
Ignoring wrong pointing object 421 0 (offset 0)
Ignoring wrong pointing object 452 0 (offset 0)
Ignoring wrong pointing object 524 0 (offset 0)
  9%|▉         | 1/11 [00:59<09:53, 59.36s/it]

c:\Users\MSI\industrix\industrix_1\IINDUSTRIX_РС_2025_5_Публичное_выступление_и_продающая_презентация_.pdf → 102 chunks


 18%|█▊        | 2/11 [01:07<04:25, 29.47s/it]

c:\Users\MSI\industrix\industrix_1\INDUSTRIX + Структура ГПН.pdf → 14 chunks


...

c:\Users\MSI\industrix\industrix_1\ИТ-Сервис_Опыт_участия_в_Индастрикс.pdf → 20 chunks


 91%|█████████ | 10/11 [04:37<00:22, 22.76s/it]

c:\Users\MSI\industrix\industrix_1\Образовательная программа.pdf → 22 chunks


100%|██████████| 11/11 [04:39<00:00, 25.37s/it]

c:\Users\MSI\industrix\industrix_1\Структура ГПН.pdf → 3 chunks
✅ Готово. Результат: embeddings3.jsonl


In [ ]:
import os
import json
import numpy as np
from uuid import uuid4
from tqdm import tqdm
from pypdf import PdfReader
from openai import OpenAI

# ---------------- CONFIG ----------------
DATA_DIR = r"c:\Users\MSI\industrix\industrix_2"
OUTPUT_FILE = "embeddings4.jsonl"

EMBED_MODEL = "text-embedding-3-large"  # 3072 dim
CHUNK_SIZE = 400
OVERLAP = 50
BATCH_SIZE = 64
# ---------------------------------------

client = OpenAI(api_key="")

# -------- TEXT LOADERS --------
def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append((i + 1, text))
    return pages

def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return [(None, f.read())]

# -------- CHUNKING --------
def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
    return chunks

# -------- OPENAI EMBEDDINGS --------
def embed_texts(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [
        np.array(e.embedding, dtype="float32").tolist()
        for e in response.data
    ]

# -------- PROCESSING --------
def process_file(path, out_f):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        pages = load_pdf(path)
    elif ext == ".txt":
        pages = load_txt(path)
    else:
        return 0

    total_chunks = 0

    for page, text in pages:
        chunks = chunk_text(text, CHUNK_SIZE, OVERLAP)
        if not chunks:
            continue

        # батчами — дешевле и стабильнее
        for i in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[i:i + BATCH_SIZE]
            vectors = embed_texts(batch)

            for chunk, vector in zip(batch, vectors):
                record = {
                    "id": str(uuid4()),
                    "vector": vector,
                    "payload": {
                        "text": chunk,
                        "source": path,
                        "page": page,
                        "model": EMBED_MODEL
                    }
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                total_chunks += 1

    return total_chunks

# -------- MAIN --------
def main():
    files = []
    for root, _, filenames in os.walk(DATA_DIR):
        for name in filenames:
            if name.endswith((".pdf", ".txt")):
                files.append(os.path.join(root, name))

    print(f"Найдено файлов: {len(files)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
        for path in tqdm(files):
            try:
                chunks = process_file(path, out_f)
                tqdm.write(f"{path} → {chunks} chunks")
            except Exception as e:
                tqdm.write(f"❌ Ошибка {path}: {e}")

    print(f"✅ Готово. Результат: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


Найдено файлов: 14


  7%|▋         | 1/14 [00:00<00:06,  1.88it/s]

c:\Users\MSI\industrix\industrix_2\2 Этап.txt → 1 chunks


 14%|█▍        | 2/14 [00:03<00:25,  2.15s/it]

c:\Users\MSI\industrix\industrix_2\AnyScanner_09_03_202511.pdf → 7 chunks


...

c:\Users\MSI\industrix\industrix_2\INDUSTRIX_РС_2025_этап_2_м8_Масштабирование_проекта.pdf → 65 chunks


 93%|█████████▎| 13/14 [04:00<00:16, 16.65s/it]

c:\Users\MSI\industrix\industrix_2\RPG для INDUSTRIX.pdf → 6 chunks


100%|██████████| 14/14 [04:04<00:00, 17.49s/it]

c:\Users\MSI\industrix\industrix_2\Преакселерационная программа 2 Этап. Артем Ненько.pdf → 10 chunks
✅ Готово. Результат: embeddings4.jsonl


In [2]:
pip install qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [5]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="https://qdrant.dev.adapstory.com",
    port=443,
    timeout=120
)

client.get_collections()


CollectionsResponse(collections=[CollectionDescription(name='presentations_industrix_openai')])

In [7]:
from qdrant_client.models import VectorParams, Distance

client.recreate_collection(
    collection_name="presentations_industrix_openai",
    vectors_config=VectorParams(
        size=3072,
        distance=Distance.COSINE
    )
)

True

In [8]:
import json

BATCH_SIZE = 100

def batched(iterable, n):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

with open("embeddings3.jsonl", "r", encoding="utf-8") as f:
    for batch in batched((json.loads(line) for line in f), BATCH_SIZE):
        client.upsert(
            collection_name="presentations_industrix_openai",
            points=batch
        )

In [11]:
client.count(
    collection_name="presentations_industrix_openai",
    exact=True
)

CountResult(count=1021)

In [10]:
import json

BATCH_SIZE = 100

def batched(iterable, n):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

with open("embeddings4.jsonl", "r", encoding="utf-8") as f:
    for batch in batched((json.loads(line) for line in f), BATCH_SIZE):
        client.upsert(
            collection_name="presentations_industrix_openai",
            points=batch
        )

In [12]:
client.delete_collection("presentations_industrix_openai")


True

In [13]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="presentations_industrix_openai",
    vectors_config=VectorParams(
        size=3072,
        distance=Distance.COSINE
    )
)


True

In [ ]:
import os
from openai import OpenAI
from qdrant_client import QdrantClient

# ---------- OPENAI ----------
client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"   # 3072
CHAT_MODEL = "gpt-4o-mini"

# ---------- QDRANT ----------
QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"
TOP_K = 10

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(
    url=QDRANT_URL,
    port=443,
    timeout=120
)


In [14]:
import numpy as np

def embed_query(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return response.data[0].embedding


In [15]:
def retrieve_context(question, top_k=TOP_K):
    vector = embed_query(question)

    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True
    )

    contexts = []

    for r in results:
        payload = r.payload or {}
        text = payload.get("text", "").strip()
        if not text:
            continue

        contexts.append({
            "score": r.score,
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source")
        })

    return contexts


In [16]:
def debug_retrieval(question):
    contexts = retrieve_context(question)

    print(f"\n🔍 Вопрос: {question}\n")
    for i, c in enumerate(contexts, 1):
       # print(f"--- TOP {i} ---")
        #print(f"Score: {c['score']:.3f}")
        #print(f"Source: {c['source']} | Page: {c['page']}")
        print(c["text"][:500])
        print()


In [17]:
def build_prompt(contexts, question):
    context_text = "\n\n".join(
        f"[стр. {c['page']} | score {c['score']:.2f}]\n{c['text']}"
        for c in contexts
    )

    return f"""
Ты — эксперт по материалам компании Industrix.
Отвечай СТРОГО на основе контекста ниже.
Если ответа в контексте нет — скажи:
"В предоставленных материалах нет информации".

КОНТЕКСТ:
{context_text}

ВОПРОС:
{question}

ОТВЕТ:
"""


In [18]:
def ask_openai(prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=800
    )
    return response.choices[0].message.content


In [19]:
def rag_answer(question):
    contexts = retrieve_context(question)

    if not contexts:
        return "В базе нет релевантной информации."

    prompt = build_prompt(contexts, question)
    return ask_openai(prompt)


In [9]:
question = "Какие причины провала стартапа?"

debug_retrieval(question)   # ← всегда сначала смотри контекст
answer = rag_answer(question)

print(answer)



🔍 Вопрос: Какие причины провала стартапа?

6Газпром нефть ЗАКОНЧИЛИСЬ ДЕНЬГИ НЕ НУЖНЫ РЫНКУ ОСНОВНЫЕ ПРИЧИНЫ ПРОВАЛА СТАРТАПОВ Based on an analysis of 101 Startup Post-Mortems 42% 29% НЕПРАВИЛЬНАЯ КОМАНДА ВЫТЕСНИЛИ КОНКУРЕНТЫ 23% 19% ЦЕНООБРАЗОВАНИЕ 18% http://bit.ly/1v73JvGwww.cbinsights.com

9Газпром нефть ЗАКОНЧИЛИСЬ ДЕНЬГИНЕ НУЖНЫ РЫНКУ ОСНОВНЫЕ ПРИЧИНЫ ПРОВАЛА СТАРТАПОВBased on an analysis of 101 Startup Post-Mortems 42%29%НЕПРАВИЛЬНАЯ КОМАНДАВЫТЕСНИЛИ КОНКУРЕНТЫ23%19%ЦЕНООБРАЗОВАНИЕ18% http://bit.ly/1v73JvGwww.cbinsights.com

7Газпром нефть ЗАКОНЧИЛИСЬ ДЕНЬГИ НЕ НУЖНЫ РЫНКУ ОСНОВНЫЕ ПРИЧИНЫ ПРОВАЛА СТАРТАПОВ Based on an analysis of 101 Startup Post-Mortems 42% 29% НЕПРАВИЛЬНАЯ КОМАНДА ВЫТЕСНИЛИ КОНКУРЕНТЫ 23% 19% ЦЕНООБРАЗОВАНИЕ 18% Вариант упаковки Вашего решения в продукт: • не интересен бизнес-заказчику, • не решает его вызовов • не создает должный экономический эффект (затраты на внедрение, управление, взаимодействия превышают возможный эффект или эффект незначителен в денеж

In [20]:
pip install -q python-telegram-bot==20.7 

Note: you may need to restart the kernel to use updated packages.


In [21]:
pip install -q nest_asyncio


Note: you may need to restart the kernel to use updated packages.


In [22]:
def rag_answer(question: str) -> str:
    contexts = retrieve_context(question)  # ← list

    if not contexts:
        return "Я не нашёл информацию в документах."

    context_text = "\n\n".join(
        c["text"] for c in contexts
    )

    prompt = f"""
Ты отвечаешь ТОЛЬКО на основе контекста ниже.

Контекст:
{context_text}

Вопрос:
{question}

Ответ:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content.strip()


In [23]:
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters,
)
import os


In [24]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Привет! Задай вопрос по материалам Industrix 📄"
    )


In [25]:
import asyncio

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    question = update.message.text

    await update.message.reply_text("⏳ Думаю...")

    loop = asyncio.get_running_loop()
    answer = await loop.run_in_executor(
        None,
        rag_answer,
        question
    )

    await update.message.reply_text(answer[:4000])  # лимит TG


In [ ]:
BOT_TOKEN = ""

import asyncio

async def main():
    app = ApplicationBuilder().token(BOT_TOKEN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    print("🚀 Telegram RAG bot started")

    await app.initialize()
    await app.start()
    await app.bot.initialize()
    await app.updater.start_polling()

# В Jupyter:
await main()



🚀 Telegram RAG bot started


Error while getting Updates: httpx.ReadError: 
Exception happened while polling for updates.
Traceback (most recent call last):
  File "c:\Users\MSI\anaconda3\Lib\site-packages\httpcore\_exceptions.py", line 10, in map_exceptions
    yield
  File "c:\Users\MSI\anaconda3\Lib\site-packages\httpcore\_backends\anyio.py", line 34, in read
    return await self._stream.receive(max_bytes=max_bytes)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\streams\tls.py", line 205, in receive
    data = await self._call_sslobject_method(self._ssl_object.read, max_bytes)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\streams\tls.py", line 147, in _call_sslobject_method
    data = await self.transport_stream.receive()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\_backends\_asyncio.py", line 1132,

In [ ]:
from telegram import Bot

BOT_TOKEN = ""

bot = Bot(BOT_TOKEN)

await bot.delete_webhook(drop_pending_updates=True)

print("✅ Webhook и очередь обновлений сброшены")


✅ Webhook и очередь обновлений сброшены


Error while getting Updates: Conflict: terminated by setWebhook request
Exception happened while polling for updates.
Traceback (most recent call last):
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 688, in _network_loop_retry
    if not await action_cb():
           ^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 384, in polling_action_cb
    raise exc
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 373, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_extbot.py", line 558, in get_updates
    updates = await super().get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\_bot.py", line 525, in decorator
    result = await func(self, *args, **kwargs)  # skipcq: PYL-E1102
             ^^^^^^^^^^^^^